# Lexicon C19 revision experiments

Two author-run retrieval workflows and rule-order sensitivity tests on the unchanged v3.2.1 release (DOI 10.5281/zenodo.22207277). These are reproducibility demonstrations, not independent user tests or new annotation. Put the downloaded Zenodo ZIP beside this notebook, or set ARCHIVE to its location. Requires Python 3.10+, pandas, NumPy, Matplotlib, and Jupyter. The standalone audit_release.py uses only the standard library.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import audit_release
from prepare_revision_assets import figures

filename = "Lexicon_C19_Export_Package_v3.2.1_2026-08-31.zip"
candidates = [Path(filename)] + [parent / "zenodo_export_v3.2.1" / filename for parent in Path.cwd().parents]
ARCHIVE = next((p for p in candidates if p.is_file()), None)
assert ARCHIVE is not None, "Download the v3.2.1 Zenodo ZIP and set ARCHIVE."
OUTPUT = Path("notebook_diagnostics")


## Verify the frozen release and rerun the tests

The baseline must reproduce every deposited category/subcategory label before any perturbation is interpreted. The CSV and JSONL remain unchanged. The following cell reruns the rule matching and 100 seeded order permutations; it may take about a minute.

In [ ]:
evidence = audit_release.main(ARCHIVE, OUTPUT)
assert evidence["archive_md5"] == "18cc1f3ec73d8602318dab35a2cac571"
assert evidence["baseline_category_and_subcategory_mismatches"] == 0
{k: evidence[k] for k in ["selected_records", "complete_candidates", "keyword_rule_count", "top_level_collision_records", "label_pair_collision_records"]}

## Workflow A Browse lute-related candidates in the selected index

Use the non-empty language cells of the lute alignment row as exact word-boundary queries in headword plus definition. The alignment is a browsing list, not a validated translation resource. Inspect the German Alphabēt result: Laute can denote speech sounds rather than the instrument. Return to the provider source before historical quotation.

In [ ]:
a = pd.read_csv(OUTPUT / "lute_browsing_from_selected.csv")
display(a.groupby("source_dictionary").size().rename("returned_candidates").to_frame())
display(a.loc[a["id"].isin(["DE-MEYERS-1885-00036", "DE-MEYERS-1885-00243"]), ["id", "headword", "definition", "source_file", "source_line_start"]])

## Workflow B Search beyond the selected index

The broader musician query searches all parsed candidates, including GENERAL/OTHER. It does not automatically promote them to relevant records. Compare the Kunstpfeifer example with the original selection policy. Missing entries from the printed dictionaries remain outside this parsed pool.

In [ ]:
b = pd.read_csv(OUTPUT / "musician_query_in_complete_pool.csv")
display(b.groupby("in_selected_index").size().rename("returned_candidates").to_frame())
display(b.loc[b["id"] == "DE-MEYERS-1885-00235", ["id", "headword", "definition", "category", "source_file", "source_line_start"]])
evidence["retrieval_demonstrations"]["musician_query_in_complete_pool"]["query_terms"]

## Order sensitivity

Reversal and random shuffles change labels, not selection membership. Top-level categories and category/subcategory pairs are reported separately. The 100 shuffles are a deterministic stress test, not a sampling confidence interval or an accuracy benchmark.

In [ ]:
sensitivity = pd.read_csv(OUTPUT / "order_sensitivity.csv")
display(sensitivity.iloc[:1])
display(sensitivity.iloc[1:][["top_level_changed", "label_pair_changed", "subcategory_only_changed"]].agg(["min", "median", "max"]))
assert sensitivity["selected_membership_changed"].eq(0).all()

## Companion metadata and relation construction

The 27 alignment rows come from three JSON mappings and 24 fixed extras. The 527 SAME_FAMILY edges enumerate pairs within predefined groups; their counts and weights are not corpus observations. The 12 cross-domain suggestions have no per-edge source evidence and should not be used as historical findings.

In [ ]:
display(pd.Series(evidence["alignment_populated_cells_by_language"], name="populated_cells").to_frame())
display(pd.Series(evidence["relation_types"], name="rows").to_frame())
groups = pd.read_csv(OUTPUT / "semantic_group_sizes.csv")
display(groups[["group", "term_count", "expected_pair_count", "observed_pair_count"]])
assert groups["expected_pair_count"].sum() == groups["observed_pair_count"].sum() == 527
assert evidence["same_family_expected_from_group_pairs"] == 527

## Release coverage and processed-file provenance

These tables are reconstructed from the frozen release, not from newly acquired web pages. The coverage flag identifies processed source traditions; the older Fetis inventory entry is not counted as the later edition. The 19-file table separates recorded inventory identifiers from filename hints. A matching name is not proof of upstream byte identity. Historical retrieval dates, OCR-engine versions and Wikisource revision IDs remain unrecorded. Current revision IDs must not be substituted for acquisition history.

In [ ]:
coverage = pd.read_csv(OUTPUT / "inventory_release_coverage.csv")
provenance = pd.read_csv(OUTPUT / "processed_file_provenance.csv", keep_default_na=False)
display(coverage.loc[coverage["present_in_release"], ["inventory_id", "declaration_file", "complete_candidates", "selected_records"]])
display(provenance[["source_file", "recorded_inventory_ia_item", "filename_ia_item_hint", "locator_status"]])
assert coverage["complete_candidates"].sum() == provenance["complete_candidates"].sum() == 25186
assert coverage["selected_records"].sum() == provenance["selected_records"].sum() == 3760
assert len(provenance) == 19
assert provenance["historical_retrieval_date"].eq("not_recorded").all()
assert provenance["upstream_revision_id"].eq("not_recorded").all()

## Publication figures

The figures are regenerated from the same frozen tables, with larger labels. The one-record Vieira category profile is masked; the record itself is not removed. These refreshed graphics belong to the revision supplement, not to the earlier Zenodo archive.

In [ ]:
from IPython.display import Image, display
figures(ARCHIVE, OUTPUT / "figures")
for name in ["fig01_corpus_composition_and_yield", "fig02_category_profiles_heatmap"]:
    display(Image(filename=str(OUTPUT / "figures" / (name + ".png")), width=900))

## Interpretation and limits

The lute workflow retrieves 33 candidates; the broader musician query retrieves 430, of which 407 are outside the selected index. Neither result is a new relevance estimate. Reversal changes 191 broad labels and 317 full label pairs. Missing OCR engine/model provenance and absent CER/WER ground truth remain unresolved in the deposited release. No new source acquisition, human annotation, ontology formalisation, or dataset publication is performed by this notebook.